# 06 - Physics Signal-Domain Terms

This notebook isolates the signal-domain physics losses that map the
reconstructed profile back through the tomographic acquisition geometry:
`coherence_resynthesis`, `covariance_matching`, and `capon_cycle`. These
rely on the steering matrix and per-bin outer products built by
`TomoGeometry`. We confirm the geometry tensors have the expected shapes and
that each loss rises as the predicted profile is displaced from the target.

Modules exercised: `tools.tomo_geometry.TomoGeometry`,
`PhysicsComponents.coherence_resynthesis`,
`PhysicsComponents.covariance_matching`, `PhysicsComponents.capon_cycle`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
from configuration.training_config import GaussianConfig, GeometryConfig, LossConfig
from pipelines.training_pipeline.loss import Loss, PhysicsComponents as PC
from tools import NullLogger, NullTracker

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

X_MIN, X_MAX = -10.0, 40.0
x_axis       = torch.linspace(X_MIN, X_MAX, 64, dtype=torch.float32)
gaussian_cfg = GaussianConfig(n_default_gaussians=1, x_min=X_MIN, x_max=X_MAX)
geometry_cfg = GeometryConfig()

criterion = Loss(x_axis, NullLogger(), NullTracker(), gaussian_cfg,
                 LossConfig(), IdentityNormStats(), geometry_cfg)
geom  = criterion.geometry
dx    = criterion.dx
floor = 1e-3

print('tracks         :', geom.n_tracks)
print('steering shape :', tuple(geom.steering.shape), '(tracks, bins)')
print('outer shape    :', tuple(geom.outer.shape),    '(tracks, tracks, bins)')
print('kz range       : %.4f .. %.4f rad/m' % (float(geom.kz.min()), float(geom.kz.max())))


## Acquisition geometry

The interferometric phase per track is `kz * x`. We plot the real part of the
steering vectors against elevation to visualise how each baseline encodes the
height axis at a different fringe rate.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for t in range(geom.n_tracks):
    ax.plot(x_axis.numpy(), geom.steering[t].real.numpy(), lw=1, label=f'track {t}')
ax.set_xlabel('elevation x [m]')
ax.set_ylabel('Re(steering)')
ax.set_title('Steering vectors (real part) per baseline')
ax.legend(ncol=3, frameon=False, fontsize=8)
fig.tight_layout()
plt.show()


## Signal-domain losses vs peak displacement

We displace a single-peak prediction from the target and evaluate the three
signal-domain terms. Each should be near zero at zero displacement and
increase as the resynthesised coherence / covariance / Capon spectrum
diverges from the target.

In [ ]:
def curve(amp, mu, sig, H=2, W=2):
    p = torch.tensor([amp, mu, sig], dtype=torch.float32).reshape(1, 3, 1, 1)
    return criterion.reconstruct_gaussians(p.expand(1, 3, H, W).contiguous())

target  = curve(2.0, 10.0, 4.0)
shifts  = torch.linspace(0.0, 18.0, 31)
loading = criterion.loss_cfg.capon_loading

coh_v, cov_v, cap_v = [], [], []
for s in shifts:
    pred = curve(2.0, 10.0 + float(s), 4.0)
    coh_v.append(float(PC.coherence_resynthesis(pred, target, geom.steering, dx, floor)))
    cov_v.append(float(PC.covariance_matching(pred, target, geom.outer, dx, floor)))
    cap_v.append(float(PC.capon_cycle(pred, target, geom.steering, geom.outer, dx, loading, floor)))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(shifts.numpy(), coh_v); axes[0].set_title('coherence_resynthesis')
axes[1].plot(shifts.numpy(), cov_v); axes[1].set_title('covariance_matching')
axes[2].plot(shifts.numpy(), cap_v); axes[2].set_title('capon_cycle')
for ax in axes:
    ax.set_xlabel('centre shift [m]')
    ax.set_ylabel('loss value')
fig.tight_layout()
plt.show()


## Expected visual outcome

The steering plot shows oscillations whose frequency increases with the
baseline (track) index. All three signal-domain losses start near zero at
zero displacement and increase with peak shift, with the Capon term
potentially flattening once the spectra are fully decorrelated. This confirms
the geometry tensors are wired correctly and the signal-domain terms penalise
physical mismatch.